In [2]:
#%pip install ase==3.22.1
#%pip install numpy==1.24.4
#%pip install dscribe==2.5.0
#%pip install scipy

In [3]:
from ase.io import read
from ase import Atoms
import os
import glob

folder_path = './Brownmillerite/'
file_list = sorted(glob.glob(os.path.join(folder_path, '*POSCAR'))) 

atoms_list = []
stype = []
red = 0
green = 0
blue = 0
for struc in file_list:
    if "3C" in struc:
        stype.append("red")
        red += 1
    elif "6H" in struc:
        stype.append("black") 
        green += 1
    elif "12R" in struc:
        stype.append("blue")  
        blue +=1
    else:
        print("Found undefind")
        break
    atoms = read(struc,format='vasp')
    atoms_list.append(atoms)
    #print(f"Loaded: {struc}")
n_struc = len(atoms_list)
print(f"Total files loaded: {n_struc}")
print(f"3C = {red}")
print(f"6H = {green}")
print(f"12R = {blue}")
print(f"stype loaded {len(stype)} items")


Found undefind
Total files loaded: 40398
3C = 28664
6H = 63
12R = 11671
stype loaded 40398 items


In [4]:
#Parameters setting
r_cut=5
n_max=2
l_max=2
print(atoms[0])

Atom('Sr', [3.4548621409840216e-08, -0.25944047007731696, 10.185116165385109], index=0)


In [5]:
species = list(set(atoms.get_chemical_symbols()))
print(species)

['O', 'Sr', 'Co']


# Create average vector for each substance

In [6]:
import numpy as np
from dscribe.descriptors import SOAP
from ase import Atoms

energy = []
soap_vectors = []
all_atoms = []

average_soap = SOAP(
    species=species,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    average="inner",
    sparse=False,
    periodic=True
)

for cid in atoms_list:
    #print(cid)
    try:
        atoms = cid
    except KeyError:
        print(f"CID {cid} not found, skip")
        continue
    soap_vectors.append(average_soap.create(atoms))
    #all_atoms.append(atoms)
soap_array = np.array(soap_vectors)



# UMAP

In [7]:
import matplotlib.pyplot as plt

rcParams_dict = {
    # ---------- figure
    'figure.figsize': [8, 6],
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    # ---------- axes
    'axes.grid': True,
    'axes.linewidth': 1.5,
    # ---------- ticks
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 8.0,
    'ytick.major.size': 8.0,
    # ---------- lines
    'lines.linewidth': 2.5,
    'lines.markersize': 12,
    # ---------- grid
    'grid.linestyle': ':',
    # ---------- font
    'font.family': 'Times New Roman',
    'mathtext.fontset': 'cm',
    #'mathtext.fontset': 'stix',
    'font.size': 15,
    'axes.labelsize': 20,
    'legend.fontsize': 20,
    'svg.fonttype': 'path',  # Embed characters as paths
    #'svg.fonttype': 'none',  # Assume fonts are installed on the machine
    'pdf.fonttype': 42,  # embed fonts in PDF using type42 (True type)
}

plt.rcParams.update(rcParams_dict)

In [8]:
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import numpy as np

clustering = DBSCAN(eps=3, min_samples=2).fit(soap_array)
labels = clustering.labels_

# ตั้งค่าสี: -1 = noise, 0 = กลุ่ม 0, 1 = กลุ่ม 1, ...
unique_labels = set(labels)
colors = [plt.cm.Spectral(each)
          for each in np.linspace(0, 1, len(unique_labels))]

# วาดกราฟ
for k, col in zip(unique_labels, colors):
    if k == -1:
        # สีสำหรับ noise
        col = [0, 0, 0, 1]  # black

    class_member_mask = (labels == k)
    xy = soap_array[class_member_mask]

    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
             markeredgecolor='k', markersize=1)

#plt.title("DBSCAN Clustering")
##plt.xlabel("X")
#plt.ylabel("Y")
plt.grid(True)
plt.show()


In [9]:
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

centers = [[1, 1], [-1, -1], [1, -1]]
soap_array, labels_true = make_blobs( 
    n_samples=40398, centers=centers, cluster_std=0.4, random_state=0
)
print(f"{labels_true}")
soap_array = StandardScaler().fit_transform(soap_array)
import matplotlib.pyplot as plt

plt.scatter(soap_array[:, 0], soap_array[:, 1],c=stype ,cmap='Spectral', s=5)
plt.show()

[1 2 1 ... 2 1 1]


/var/folders/0f/_n97csts5c5c3ldj1k40vc100000gn/T/ipykernel_5387/4215351870.py:12: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  plt.scatter(soap_array[:, 0], soap_array[:, 1],c=stype ,cmap='Spectral', s=5)


In [ ]:
import matplotlib
matplotlib.use('TkAgg')  # or 'QtAgg'

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

centers = [[1, 1, 1], [-1, -1, -1], [1, -1, 1]]
soap_array, labels_true = make_blobs(
    n_samples=40398, centers=centers, n_features=3, cluster_std=0.4, random_state=0
)
soap_array = StandardScaler().fit_transform(soap_array)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(soap_array[:, 0], soap_array[:, 1], soap_array[:, 2],
                c=stype, cmap='Spectral', s=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

def update(angle):
    ax.view_init(elev=30, azim=angle)
    return fig,

ani = FuncAnimation(fig, update, frames=np.arange(0, 360, 2), interval=50, blit=False)

plt.show()


/var/folders/0f/_n97csts5c5c3ldj1k40vc100000gn/T/ipykernel_5387/3976040871.py:19: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  sc = ax.scatter(soap_array[:, 0], soap_array[:, 1], soap_array[:, 2],
invalid command name "5344133504process_stream_events"
    while executing
"5344133504process_stream_events"
    ("after" script)
can't invoke "event" command: application has been destroyed
    while executing
"event generate $w <<ThemeChanged>>"
    (procedure "ttk::ThemeChanged" line 6)
    invoked from within
"ttk::ThemeChanged"


invalid command name "4999490816delayed_destroy"
    while executing
"4999490816delayed_destroy"
    ("after" script)


: 